In [3]:
using ITensors, ITensorMPS, ITensorInfiniteMPS
using SparseArrays
using LinearAlgebra
using Plots
using JLD2

In [22]:
SWAP = [1 0 0 0
        0 0 1 0
        0 1 0 0
        0 0 0 1]

4×4 Matrix{Int64}:
 1  0  0  0
 0  0  1  0
 0  1  0  0
 0  0  0  1

In [26]:
function decoherence_gate(M::AbstractMatrix, p::Float64, sites::Vector{Index{Int}})
    g = (1-p)*op(I, sites...) + p*op(M, sites...)

    return g
end


decoherence_gate (generic function with 3 methods)

In [16]:
collect(siteinds(ψ))

2-element Vector{Index{Int64}}:
 (dim=2|id=95|"Qubit,Site,c=1,n=1")
 (dim=2|id=399|"Qubit,Site,c=1,n=2")

In [4]:
s = siteinds("Qubit", 2)
ψ = InfiniteMPS(s)

# Extract indices for each site tensor
l0, s1, l1 = inds(ψ[1])
l1p, s2, l2 = inds(ψ[2])

# Set ψ[1] = [1, 0] reshaped as 1×2×1
A1 = reshape([1.0, 0.0], 1, 2, 1)
ψ[1] = ITensor(A1, l0, s1, l1)

# Set ψ[2] = [0, 1] reshaped as 1×2×1
A2 = reshape([0.0, 1.0], 1, 2, 1)
ψ[2] = ITensor(A2, l1p, s2, l2)

ITensor ord=3 (dim=1|id=77|"Link,c=1,l=1") (dim=2|id=399|"Qubit,Site,c=1,n=2") (dim=1|id=887|"Link,c=1,l=2")
NDTensors.Dense{Float64, Vector{Float64}}

In [ ]:
function evolve!(ψ::InfiniteMPS, pos::Int, p::Float64; maxdim=128, cutoff=1e-8)
    s1 = siteinds(ψ[pos])[1]
    s2 = siteinds(ψ[pos+1])[1]
    g = decoherence_gate(SWAP, p, [s1, s2])

    
    l0 = inds(ψ[pos]; tags="Link")
    U, S, V = svd(g*ψ[pos]*ψ[pos+1], )

In [71]:
ψ[1]

ITensor ord=3 (dim=1|id=887|"Link,c=0,l=2") (dim=2|id=95|"Qubit,Site,c=1,n=1") (dim=1|id=77|"Link,c=1,l=1")
NDTensors.Dense{Float64, Vector{Float64}}

In [29]:
x = decoherence_gate(SWAP, 0.1, collect(siteinds(ψ))) * ψ[1] * ψ[2]

ITensor ord=4 (dim=2|id=95|"Qubit,Site,c=1,n=1")' (dim=2|id=399|"Qubit,Site,c=1,n=2")' (dim=1|id=887|"Link,c=0,l=2") (dim=1|id=887|"Link,c=1,l=2")
NDTensors.Dense{Float64, Vector{Float64}}

In [62]:
s1 = inds(ψ[1]; tags="Site")[1]
s2 = inds(ψ[2]; tags="Site")[1]
l0 = inds(ψ[1]; tags="Link,l=2")
l2 = inds(ψ[2]; tags="Link,l=1")

((dim=1|id=77|"Link,c=1,l=1"),)

In [70]:
typeof(l0)

Tuple{Index{Int64}}

In [69]:
prime(x, -1)

ITensor ord=4 (dim=2|id=95|"Qubit,Site,c=1,n=1") (dim=2|id=399|"Qubit,Site,c=1,n=2") (dim=1|id=887|"Link,c=0,l=2") (warning: prime level -1 is less than 0) (dim=1|id=887|"Link,c=1,l=2") (warning: prime level -1 is less than 0)
NDTensors.Dense{Float64, Vector{Float64}}

In [65]:
svd(x, prime(s1), l0)

ITensors.TruncSVD(ITensor ord=3
Dim 1: (dim=2|id=95|"Qubit,Site,c=1,n=1")'
Dim 2: (dim=1|id=887|"Link,c=0,l=2")
Dim 3: (dim=2|id=821|"Link,u")
NDTensors.Dense{Float64, Vector{Float64}}
 2×1×2
[:, :, 1] =
 -1.0
  0.0

[:, :, 2] =
  0.0
 -1.0
, ITensor ord=2
Dim 1: (dim=2|id=821|"Link,u")
Dim 2: (dim=2|id=506|"Link,v")
NDTensors.Diag{Float64, Vector{Float64}}
 2×2
 0.9  0.0
 0.0  0.1
, ITensor ord=3
Dim 1: (dim=2|id=399|"Qubit,Site,c=1,n=2")'
Dim 2: (dim=1|id=887|"Link,c=1,l=2")
Dim 3: (dim=2|id=506|"Link,v")
NDTensors.Dense{Float64, Vector{Float64}}
 2×1×2
[:, :, 1] =
 -0.0
 -1.0

[:, :, 2] =
 -1.0
 -0.0
, Spectrum{Vector{Float64}, Float64}([0.81, 0.010000000000000002], 0.0), (dim=2|id=821|"Link,u"), (dim=2|id=506|"Link,v"))

In [60]:
y = svd(x, [prime(inds(ψ[1]; tags="Site")[1]), inds(ψ[1]; tags="Link,l=2"), prime(inds(ψ[2]; tags="Site")[1])], [inds(ψ[2]; tags="Link,l=1")]; maxdim=10)

ITensors.TruncSVD(ITensor ord=3
Dim 1: (dim=2|id=95|"Qubit,Site,c=1,n=1")'
Dim 2: (dim=2|id=399|"Qubit,Site,c=1,n=2")'
Dim 3: (dim=1|id=684|"Link,u")
NDTensors.Dense{Float64, Vector{Float64}}
 2×2×1
[:, :, 1] =
 0.0                  0.9938837346736189
 0.11043152607484655  0.0
, ITensor ord=2
Dim 1: (dim=1|id=684|"Link,u")
Dim 2: (dim=1|id=254|"Link,v")
NDTensors.Diag{Float64, Vector{Float64}}
 1×1
 0.9055385138137417
, ITensor ord=3
Dim 1: (dim=1|id=887|"Link,c=0,l=2")
Dim 2: (dim=1|id=887|"Link,c=1,l=2")
Dim 3: (dim=1|id=254|"Link,v")
NDTensors.Dense{Float64, Vector{Float64}}
 1×1×1
[:, :, 1] =
 1.0
, Spectrum{Vector{Float64}, Float64}([0.8200000000000002], 0.0), (dim=1|id=684|"Link,u"), (dim=1|id=254|"Link,v"))

In [ ]:
ITensors.svd()

svd (generic function with 23 methods)

In [ ]:
y.U

ITensor ord=3 (dim=2|id=95|"Qubit,Site,c=1,n=1")' (dim=2|id=399|"Qubit,Site,c=1,n=2")' (dim=1|id=251|"Link,u")
NDTensors.Dense{Float64, Vector{Float64}}

In [56]:
y.S * y.V

ITensor ord=3 (dim=1|id=251|"Link,u") (dim=1|id=887|"Link,c=0,l=2") (dim=1|id=887|"Link,c=1,l=2")
NDTensors.Dense{Float64, Vector{Float64}}

In [44]:
inds(ψ[1]; tags="Link,l=2")

((dim=1|id=887|"Link,c=0,l=2"),)